In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [2]:
# Load the dataset
df = pd.read_csv('PRSA_data_2010.1.1-2014.12.31.csv')


print("shape: ", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

shape:  (43824, 13)

Column names: ['No', 'year', 'month', 'day', 'hour', 'pm2.5', 'DEWP', 'TEMP', 'PRES', 'cbwd', 'Iws', 'Is', 'Ir']

First 5 rows:


,No,year,month,day,hour,pm2.5,DEWP,TEMP,PRES,cbwd,Iws,Is,Ir
0,1,2010,1,1,0,NaN,-21,-11.0,1021.0,NW,1.79,0,0
1,2,2010,1,1,1,NaN,-21,-12.0,1020.0,NW,4.92,0,0
2,3,2010,1,1,2,NaN,-21,-11.0,1019.0,NW,6.71,0,0
3,4,2010,1,1,3,NaN,-21,-14.0,1019.0,NW,9.84,0,0
4,5,2010,1,1,4,NaN,-20,-12.0,1018.0,NW,12.97,0,0


In [3]:
#clean and preprocess the data


df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])

# Step 2: Set datetime as the index
df = df.set_index('datetime')

# Step 3: Keep only the columns we need
df = df[['pm2.5', 'TEMP', 'PRES', 'Iws']]

# Step 4: Rename for convenience
df.columns = ['pm25', 'temp', 'pressure', 'wind']

# Step 5: Check how many missing values we have
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal rows: {len(df)}")


Missing values per column:
pm25        2067
temp           0
pressure       0
wind           0
dtype: int64

Total rows: 43824


In [4]:
df = df.fillna(method='ffill')

# Step 2: Check no missing values remain
print("Missing values after filling:")
print(df.isnull().sum())

# Step 3: Resample hourly data to daily averages
df_daily = df.resample('D').mean()

print(f"\nHourly data points: {len(df)}")
print(f"Daily data points:  {len(df_daily)}")
print("\nFirst 5 daily rows:")
df_daily.head()

Missing values after filling:
pm25        24
temp         0
pressure     0
wind         0
dtype: int64

Hourly data points: 43824
Daily data points:  1826

First 5 daily rows:


,pm25,temp,pressure,wind
datetime,,,,
2010-01-01,NaN,-6.750000,1017.083333,14.458333
2010-01-02,145.958333,-5.125000,1024.750000,24.860000
2010-01-03,78.833333,-8.541667,1022.791667,70.937917
2010-01-04,31.333333,-11.500000,1029.291667,111.160833
2010-01-05,42.458333,-14.458333,1033.625000,56.920000


In [5]:
df_daily = df_daily.bfill()

# Confirm everything is clean
print("Missing values after backward fill:")
print(df_daily.isnull().sum())
print(f"\nDate range: {df_daily.index[0]} to {df_daily.index[-1]}")
print(f"\nBasic statistics:")
df_daily.describe().round(2)

Missing values after backward fill:
pm25        0
temp        0
pressure    0
wind        0
dtype: int64

Date range: 2010-01-01 00:00:00 to 2014-12-31 00:00:00

Basic statistics:


,pm25,temp,pressure,wind
count,1826.00,1826.00,1826.00,1826.00
mean,97.81,12.45,1016.45,23.89
std,76.96,11.56,10.07,41.36
min,2.96,-14.46,994.04,1.41
25%,41.65,1.54,1007.94,5.91
50%,78.56,13.90,1016.23,10.95
75%,131.04,23.17,1024.53,22.23
max,541.04,32.88,1043.46,463.19


In [6]:
pip install kaleido

Note: you may need to restart the kernel to use updated packages.


In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

variables = ['pm25', 'temp', 'pressure', 'wind']
colors    = ['#1a5276', '#117a65', '#e67e22', '#8e44ad']
labels    = ['PM2.5 (μg/m³)', 'Temperature (°C)', 'Pressure (hPa)', 'Wind Speed (m/s)']

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=labels,
    vertical_spacing=0.06
)

for i, (var, color, label) in enumerate(zip(variables, colors, labels), start=1):
    # Raw daily signal
    fig.add_trace(go.Scatter(
        x=df_daily.index,
        y=df_daily[var],
        mode='lines',
        name=f'{label} (daily)',
        line=dict(color=color, width=0.8),
        opacity=0.4,
        legendgroup=var,
    ), row=i, col=1)

    # 30-day rolling mean
    rolling = df_daily[var].rolling(window=30, center=True).mean()
    fig.add_trace(go.Scatter(
        x=df_daily.index,
        y=rolling,
        mode='lines',
        name=f'{label} (30d mean)',
        line=dict(color=color, width=2.5),
        legendgroup=var,
    ), row=i, col=1)

fig.update_layout(
    height=900,
    title=dict(
        text='Beijing Air Quality — Daily Averages (2010–2014)<br>'
             '<sup>Zoom, pan, or hover over any point to explore</sup>',
        font=dict(size=16)
    ),
    hovermode='x unified',   # shows all 4 values when you hover on a date
    template='plotly_white',
    legend=dict(orientation='h', y=-0.05)
)

# Save as interactive HTML file you can open in any browser
fig.write_html('beijing_interactive.html')
fig.write_image('beijing_raw.png', width=1400, height=900, scale=2)

fig.show()
print("Interactive chart saved as beijing_interactive.html!")

Interactive chart saved as beijing_interactive.html!
